# 06_rnn_lstm_gru: Sequence Classifier Comparison in PyTorch

This notebook constructs a simple bidirectional sequence classifier in PyTorch, comparing RNN, LSTM, and GRU architectures.


In [1]:
import torch
import torch.nn as nn

# Model parameters
vocab_size = 100
embedding_dim = 16
hidden_dim = 32
num_classes = 2

# Mock input batch: size=2, sequence_length=5
x = torch.randint(0, vocab_size, (2, 5))

# 1. Define models
class RecurrentClassifier(nn.Module):
    def __init__(self, cell_type="RNN"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        if cell_type == "RNN":
            self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        elif cell_type == "LSTM":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        elif cell_type == "GRU":
            self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
            
        # bidirectional outputs are doubled in dimensionality
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.rnn(embedded)
        # Take the output of the final time step
        last_step = out[:, -1, :]
        logits = self.fc(last_step)
        return logits

# 2. Run forward pass
for cell_name in ["RNN", "LSTM", "GRU"]:
    model = RecurrentClassifier(cell_type=cell_name)
    output = model(x)
    print(f"[{cell_name} Classifier] Output Logits Shape: {output.shape}")


[RNN Classifier] Output Logits Shape: torch.Size([2, 2])
[LSTM Classifier] Output Logits Shape: torch.Size([2, 2])
[GRU Classifier] Output Logits Shape: torch.Size([2, 2])


### Output Explanation
- The PyTorch modules define recurrent networks.
- Setting `bidirectional=True` concatenates the forward and backward hidden state vectors, outputting a vector of dimension `hidden_dim * 2` before classification.
